# Neural Volatility Surface — Google Colab Training & Evaluation Lab

**Purpose:** run the Neural Volatility Surface research workflow in Google Colab rather than on your local CPU.

This notebook is designed to:
- ingest a licensed options dataset from CSV/Google Drive, or generate clearly labelled **synthetic demo data**;
- validate and clean option quotes;
- calculate implied volatility from prices when needed;
- engineer volatility-surface and market features;
- construct a leakage-aware next-observation forecasting target;
- compare persistence, statistical/ML baselines, and an optional PyTorch neural network;
- evaluate chronologically with a holdout period and report error by surface region;
- save the model, metrics, and predictions.

## Important research boundary
The project's current evidence is synthetic. Cboe was discussed as a *candidate* data provider, but access, licensing, coverage, and price have not been confirmed. This notebook does **not** download or claim access to Cboe data. For real-market research, upload a dataset you are authorized to use. Synthetic demo results are plumbing checks, not evidence of real-market predictive performance.

## Cavemen + Ponytail
Keep the workflow simple, reproducible, and cheap: reuse a small dependency set, start with a small run, compare against persistence, avoid unnecessary architecture, and do not select a model using the final test set.

## 0. Colab setup

**GPU-only policy for neural computation.** In Colab choose **Runtime → Change runtime type → GPU** (T4 or better). All neural training, validation, evaluation and inference run on the explicitly selected CUDA device (`GPU_INDEX`). The notebook verifies the device before any neural work and **stops with an error if no supported GPU is present - it never falls back to the CPU**. Locally, the designated GPU is the NVIDIA RTX 4060 (8 GB) with a CUDA build of PyTorch. Light scikit-learn baselines and preprocessing run on the CPU. Set `RUN_NEURAL = False` in the next code cell only if you deliberately want baselines without any neural result.

For real data, mount Google Drive or upload a CSV in the data-ingestion cell. Do not upload data you are not permitted to use.


In [ ]:
# Install only the small set of packages used by the notebook.
%pip -q install numpy pandas scipy scikit-learn matplotlib joblib

In [ ]:
import os, sys, json, math, random, time, warnings, platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import brentq
from scipy.stats import norm
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- GPU-only policy for neural computation (no CPU fallback) ---
RUN_NEURAL = True            # False = scikit-learn baselines only; no neural result is produced (explicit choice, not a fallback)
GPU_INDEX = 0                # explicitly selected CUDA device
EXPECTED_LOCAL_GPU = "4060"  # designated local GPU (NVIDIA RTX 4060 8 GB); any Colab GPU is accepted when running in Colab
IN_COLAB = "google.colab" in sys.modules
torch, DEVICE, HW_META = None, None, {}


def require_gpu():
    """Verify the selected CUDA device and return (torch.device, hardware metadata). Raises RuntimeError; never uses the CPU."""
    if not torch.cuda.is_available():
        raise RuntimeError(("No CUDA GPU available (torch %s, CUDA build %s). Colab: Runtime > Change runtime type > GPU. "
                            "Local: install a CUDA build of PyTorch for the RTX 4060. Neural computation is GPU-only; stopping.") %
                           (torch.__version__, torch.version.cuda))
    if GPU_INDEX >= torch.cuda.device_count():
        raise RuntimeError("GPU_INDEX=%d but only %d CUDA device(s) are visible." % (GPU_INDEX, torch.cuda.device_count()))
    name = torch.cuda.get_device_name(GPU_INDEX)
    if not IN_COLAB and EXPECTED_LOCAL_GPU not in name:
        raise RuntimeError("Local neural runs must use the designated RTX 4060; found %r." % name)
    dev = torch.device("cuda", GPU_INDEX)
    torch.cuda.set_device(dev)
    torch.ones(8, device=dev).sum().item()   # prove the device executes work
    torch.cuda.synchronize(dev)
    p = torch.cuda.get_device_properties(dev)
    return dev, {
        "recorded_utc": datetime.now(timezone.utc).isoformat(), "python": platform.python_version(), "platform": platform.platform(),
        "in_colab": IN_COLAB, "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(), "gpu_index": GPU_INDEX, "gpu_name": p.name,
        "gpu_total_memory_gib": round(p.total_memory / 2**30, 2), "compute_capability": "%d.%d" % (p.major, p.minor),
        "seed": SEED, "numpy": np.__version__, "pandas": pd.__version__,
    }


if RUN_NEURAL:
    import torch                       # a missing PyTorch is an error, not a silent skip
    DEVICE, HW_META = require_gpu()    # fail fast before any neural computation
    torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("Neural device:", HW_META if RUN_NEURAL else "none (RUN_NEURAL=False)")


## 1. Choose data mode

- `synthetic`: creates a small artificial panel so you can run every cell and check the pipeline.
- `csv`: load a CSV from Drive or `/content/`.

**Suggested CSV schema:** one row per option quote observation, with at least:
`date, underlying, spot, strike, expiry, option_type` and either `iv` (implied volatility as decimal, e.g. `0.25`) or `bid, ask` (option prices in the same currency units as `spot`). Optional columns: `rate, dividend_yield, volume, open_interest`.

Dates must be observation dates; `expiry` must be expiration dates. `option_type` should be `call`/`put` or `C`/`P`. If your provider uses different column names, map them in `COLUMN_MAP` below.

In [ ]:
# --- USER CONFIG ---
DATA_MODE = "synthetic"   # change to "csv" for your authorized real options dataset
CSV_PATH = "/content/options.csv"  # e.g. "/content/drive/MyDrive/options/options.csv"
OUTPUT_DIR = Path("/content/nvs_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Map your source's column names to canonical names. Edit only as needed.
COLUMN_MAP = {
    # "quote_date": "date",
    # "underlying_symbol": "underlying",
    # "underlying_price": "spot",
    # "expiration": "expiry",
    # "implied_volatility": "iv",
}

### 1A. Optional: mount Google Drive or upload a CSV

Run only one of these if using real data. Do not share API keys or credentials in notebook cells.

In [ ]:
# Option A: Google Drive
# from google.colab import drive
# drive.mount('/content/drive')

# Option B: upload a file from your computer
# from google.colab import files
# uploaded = files.upload()
# CSV_PATH = "/content/" + next(iter(uploaded))

## 2. Synthetic demo generator

This generator is deliberately modest so it is safe to run in Colab. It creates artificial IV observations with temporal persistence, moneyness skew, term structure, and noise. It is **not** a calibrated market simulator and should not be used to claim real-world performance.

In [ ]:
def make_synthetic_panel(n_days=180, n_strikes=9, seed=SEED):
    """Generate a small artificial panel with stable contract identifiers over time."""
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2023-01-02", periods=n_days)
    initial_spot = 100.0
    spot = initial_spot
    level = 0.24
    rows = []

    # Fixed strikes and fixed calendar expiries allow the same contract to be matched over dates.
    strike_grid = np.round(initial_spot * np.linspace(0.75, 1.25, n_strikes), 2)
    expiry_candidates = sorted({
        (dates[0] + pd.Timedelta(days=offset)).normalize()
        for offset in range(30, n_days + 120, 30)
    })

    for di, date in enumerate(dates):
        spot *= np.exp(rng.normal(0.0002, 0.012))
        level = np.clip(0.96 * level + 0.04 * 0.23 + rng.normal(0, 0.004), 0.08, 0.8)
        expiries = [e for e in expiry_candidates if e > date and 0 < (e-date).days <= 120]
        for expiry in expiries:
            tau = max((expiry - date).days / 365.0, 1/365)
            for strike in strike_grid:
                m = np.log(strike / spot)
                for opt_type in ("call", "put"):
                    skew = 0.12 * m + 0.10 * m*m
                    term = 0.015 * np.sqrt(tau)
                    iv = float(np.clip(level + skew + term + rng.normal(0, 0.006), 0.05, 1.2))
                    # Synthetic mid/spread are diagnostics only; IV is generated directly.
                    mid = max(0.01, spot * iv * np.sqrt(tau) * 0.18)
                    spread = max(0.01, mid * rng.uniform(0.01, 0.08))
                    rows.append({
                        "date": date, "underlying": "SYNTH", "spot": spot,
                        "strike": float(strike), "expiry": expiry,
                        "option_type": opt_type, "iv": iv,
                        "bid": max(0.0, mid-spread/2), "ask": mid+spread/2,
                        "rate": 0.02, "dividend_yield": 0.0,
                        "volume": int(rng.integers(0, 500)),
                        "open_interest": int(rng.integers(0, 5000)),
                        "is_synthetic": True
                    })
    return pd.DataFrame(rows)

def load_panel():
    if DATA_MODE == "synthetic":
        df = make_synthetic_panel()
        print("Using SYNTHETIC demo data — not real-market validation.")
    elif DATA_MODE == "csv":
        path = Path(CSV_PATH)
        if not path.exists():
            raise FileNotFoundError(f"CSV not found: {path}. Upload/mount your authorized dataset and update CSV_PATH.")
        df = pd.read_csv(path)
        df = df.rename(columns=COLUMN_MAP)
        df["is_synthetic"] = False
    else:
        raise ValueError("DATA_MODE must be 'synthetic' or 'csv'")
    return df

raw = load_panel()
print("Rows:", len(raw), "| columns:", list(raw.columns))
raw.head()

## 3. Canonicalize and validate the dataset

We standardize types, remove impossible rows and crossed/missing quotes where quote prices are available, and compute basic quote diagnostics. This is not a substitute for reviewing your data provider's contract definitions and market conventions.

In [ ]:
def canonicalize_quotes(df):
    df = df.copy()
    required = ["date", "underlying", "spot", "strike", "expiry", "option_type"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["expiry"] = pd.to_datetime(df["expiry"], errors="coerce").dt.normalize()
    for c in ["spot", "strike", "iv", "bid", "ask", "rate", "dividend_yield", "volume", "open_interest"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["option_type"] = df["option_type"].astype(str).str.lower().replace({"c":"call", "p":"put"})
    df["underlying"] = df["underlying"].astype(str)
    if "rate" not in df: df["rate"] = 0.0
    if "dividend_yield" not in df: df["dividend_yield"] = 0.0
    if "volume" not in df: df["volume"] = np.nan
    if "open_interest" not in df: df["open_interest"] = np.nan
    if "is_synthetic" not in df: df["is_synthetic"] = False

    df = df.dropna(subset=required + ["spot", "strike"])
    df = df[(df["spot"] > 0) & (df["strike"] > 0) & (df["expiry"] > df["date"])]
    df = df[df["option_type"].isin(["call", "put"])]
    if {"bid", "ask"}.issubset(df.columns):
        df = df[(df["bid"].isna() | (df["bid"] >= 0)) & (df["ask"].isna() | (df["ask"] > 0))]
        df = df[df["bid"].isna() | df["ask"].isna() | (df["ask"] >= df["bid"])]
        df["mid"] = (df["bid"] + df["ask"]) / 2
        df["relative_spread"] = (df["ask"] - df["bid"]) / df["mid"].replace(0, np.nan)
    else:
        df["mid"] = np.nan
        df["relative_spread"] = np.nan
    df["tau"] = (df["expiry"] - df["date"]).dt.days / 365.0
    df["log_moneyness"] = np.log(df["strike"] / df["spot"])
    df["is_call"] = (df["option_type"] == "call").astype(int)
    df["day_of_week"] = df["date"].dt.dayofweek
    return df.sort_values(["underlying", "date", "expiry", "strike", "option_type"]).reset_index(drop=True)

quotes = canonicalize_quotes(raw)
print(f"Rows after basic validation: {len(quotes):,}")
print("Date range:", quotes.date.min(), "to", quotes.date.max())
print("Synthetic rows:", int(quotes.is_synthetic.sum()), "/", len(quotes))
quotes.head()

## 4. Implied volatility calculation (if the dataset does not already provide it)

If `iv` is supplied, it is used after basic range checks. If not, the notebook attempts to invert Black–Scholes from `mid`. The inversion is European-option based and assumes supplied `rate` and `dividend_yield`; unsuitable quote conventions, American options, stale quotes, and poor bid/ask quality can produce unreliable IV. Review those assumptions for your dataset.

In [ ]:
def bs_price(spot, strike, tau, rate, q, sigma, option_type):
    if tau <= 0 or sigma <= 0:
        return max(spot-strike, 0.0) if option_type == "call" else max(strike-spot, 0.0)
    vol_sqrt_t = sigma * np.sqrt(tau)
    d1 = (np.log(spot/strike) + (rate-q+0.5*sigma*sigma)*tau) / vol_sqrt_t
    d2 = d1 - vol_sqrt_t
    if option_type == "call":
        return spot*np.exp(-q*tau)*norm.cdf(d1) - strike*np.exp(-rate*tau)*norm.cdf(d2)
    return strike*np.exp(-rate*tau)*norm.cdf(-d2) - spot*np.exp(-q*tau)*norm.cdf(-d1)

def implied_vol_from_mid(row):
    if not np.isfinite(row.get("mid", np.nan)) or row["mid"] <= 0:
        return np.nan
    args = (row["spot"], row["strike"], row["tau"], row["rate"], row["dividend_yield"])
    price = row["mid"]
    try:
        return brentq(lambda sigma: bs_price(*args, sigma, row["option_type"]) - price,
                      1e-4, 3.0, maxiter=100)
    except (ValueError, RuntimeError, OverflowError):
        return np.nan

quotes["iv_source"] = "provided"
if "iv" not in quotes.columns:
    quotes["iv"] = np.nan
needs_iv = quotes["iv"].isna() | (quotes["iv"] <= 0) | (quotes["iv"] > 3.0)
can_invert = needs_iv & quotes["mid"].notna()
quotes.loc[can_invert, "iv_source"] = "black_scholes_inverted"
quotes.loc[can_invert, "iv"] = quotes.loc[can_invert].apply(implied_vol_from_mid, axis=1)
quotes.loc[~needs_iv, "iv"] = quotes.loc[~needs_iv, "iv"].clip(0.001, 3.0)
quotes = quotes[quotes["iv"].notna() & quotes["iv"].between(0.001, 3.0)].copy()

if quotes.empty:
    raise ValueError("No valid IV rows remain. Check the source IV units (use decimal, e.g. 0.25 not 25) or quote prices.")
print("Rows with usable IV:", len(quotes))
print(quotes["iv_source"].value_counts(dropna=False))

## 5. Build the next-observation target without look-ahead leakage

The target is the implied volatility of the **same underlying / expiry / strike / option type on the next available observation date**. We sort chronologically and merge to the next date for that contract. Expired or unmatched contracts are excluded. This is a one-observation-ahead quote-level forecasting task; it is not automatically a fixed one-day forecast for irregularly sampled data.

For real data, check that each date represents a consistent observation time (e.g. end-of-day). If the feed contains intraday records, aggregate to a consistent sampling interval before training.

In [ ]:
CONTRACT_KEYS = ["underlying", "expiry", "strike", "option_type"]
quotes = quotes.sort_values(["underlying", "expiry", "strike", "option_type", "date"]).copy()
quotes["target_date"] = quotes.groupby(CONTRACT_KEYS)["date"].shift(-1)
quotes["target_iv"] = quotes.groupby(CONTRACT_KEYS)["iv"].shift(-1)
quotes["target_spot"] = quotes.groupby(CONTRACT_KEYS)["spot"].shift(-1)

# Drop last quote for each contract; target rows must be from a strictly later date.
dataset = quotes[quotes["target_date"].notna() & (quotes["target_date"] > quotes["date"])].copy()
dataset["target_log_return"] = np.log(dataset["target_spot"] / dataset["spot"])
dataset["iv_change"] = dataset["target_iv"] - dataset["iv"]

# Causal historical features, grouped by contract.
dataset["iv_lag1"] = dataset["iv"]
dataset["iv_change_lag1"] = dataset.groupby(CONTRACT_KEYS)["iv"].diff()
dataset["spot_return_1"] = dataset.groupby("underlying")["spot"].transform(lambda s: np.log(s).diff())
dataset["iv_rolling_mean_5"] = dataset.groupby(CONTRACT_KEYS)["iv"].transform(lambda s: s.rolling(5, min_periods=2).mean())
dataset["iv_rolling_std_5"] = dataset.groupby(CONTRACT_KEYS)["iv"].transform(lambda s: s.rolling(5, min_periods=2).std())

# Market/surface features available at the current observation only.
dataset["log_tau"] = np.log(dataset["tau"].clip(lower=1/365))
dataset["abs_log_moneyness"] = dataset["log_moneyness"].abs()
dataset["moneyness_x_tau"] = dataset["log_moneyness"] * dataset["tau"]
dataset["iv_x_moneyness"] = dataset["iv"] * dataset["log_moneyness"]
dataset["iv_x_sqrt_tau"] = dataset["iv"] * np.sqrt(dataset["tau"])
dataset["log_volume"] = np.log1p(dataset["volume"].clip(lower=0))
dataset["log_open_interest"] = np.log1p(dataset["open_interest"].clip(lower=0))
dataset["relative_spread"] = dataset["relative_spread"].replace([np.inf, -np.inf], np.nan)

print("Forecast examples:", len(dataset))
print("Target dates:", dataset.date.min(), "to", dataset.target_date.max())
dataset.head()

## 6. Quality checks and exploratory plots

These checks help identify data problems; they do not certify a dataset as arbitrage-free or economically valid.

In [ ]:
display(dataset[["iv", "tau", "log_moneyness", "relative_spread", "target_iv"]].describe(percentiles=[.01,.05,.5,.95,.99]).T)

fig, ax = plt.subplots(figsize=(10, 4))
daily = dataset.groupby("date")["iv"].median()
ax.plot(daily.index, daily.values)
ax.set_title("Median observed implied volatility by date")
ax.set_ylabel("IV (decimal)")
ax.grid(True, alpha=.3)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
sample = dataset.sample(min(5000, len(dataset)), random_state=SEED)
sc = ax.scatter(sample["log_moneyness"], sample["tau"], c=sample["iv"], s=8)
ax.set_xlabel("Log-moneyness log(K/S)")
ax.set_ylabel("Time to expiry (years)")
ax.set_title("Observed IV surface samples")
plt.colorbar(sc, ax=ax, label="IV")
plt.show()

print("Rows with IV outside [0.01, 2.0]:", int((~dataset.iv.between(.01, 2.0)).sum()))
print("Rows with relative spread > 50%:", int((dataset.relative_spread > .5).sum()))

## 7. Chronological train / validation / test split

No random row split: rows from later dates must not leak into training. We split on the **target date** so the target observation's date determines its partition. This simple holdout is the first evaluation layer; a full walk-forward backtest is recommended before drawing research conclusions.

In [ ]:
FEATURES = [
    "iv", "iv_lag1", "iv_change_lag1", "spot_return_1",
    "iv_rolling_mean_5", "iv_rolling_std_5",
    "tau", "log_tau", "log_moneyness", "abs_log_moneyness",
    "moneyness_x_tau", "iv_x_moneyness", "iv_x_sqrt_tau",
    "is_call", "day_of_week", "rate", "dividend_yield",
    "relative_spread", "log_volume", "log_open_interest",
]
TARGET = "target_iv"

# Chronological date boundaries (70% train, 15% validation, 15% test by unique target dates).
unique_dates = np.array(sorted(dataset["target_date"].dropna().unique()))
if len(unique_dates) < 10:
    raise ValueError("Too few unique target dates for a meaningful chronological split.")
d1 = unique_dates[int(len(unique_dates)*0.70)]
d2 = unique_dates[int(len(unique_dates)*0.85)]

train = dataset[dataset["target_date"] < d1].copy()
valid = dataset[(dataset["target_date"] >= d1) & (dataset["target_date"] < d2)].copy()
test = dataset[dataset["target_date"] >= d2].copy()

print("Train:", len(train), train.target_date.min(), "to", train.target_date.max())
print("Valid:", len(valid), valid.target_date.min(), "to", valid.target_date.max())
print("Test :", len(test), test.target_date.min(), "to", test.target_date.max())
assert train.target_date.max() < valid.target_date.min()
assert valid.target_date.max() < test.target_date.min()

## 8. Baselines and machine-learning models

Persistence predicts next IV as current IV. This is a strong and essential baseline for persistent financial series. We also fit Ridge regression and histogram gradient boosting. Hyperparameters are intentionally modest. Validation data is for model selection; test data is used once for final comparison.

In [ ]:
def metric_row(name, y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    err = y_pred - y_true
    return {
        "model": name,
        "n": len(y_true),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "Bias_pred_minus_true": float(np.mean(err)),
        "R2": r2_score(y_true, y_pred) if len(y_true) > 1 else np.nan,
    }

X_train, y_train = train[FEATURES], train[TARGET]
X_valid, y_valid = valid[FEATURES], valid[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

imputer = SimpleImputer(strategy="median")
Xtr = imputer.fit_transform(X_train)
Xva = imputer.transform(X_valid)
Xte = imputer.transform(X_test)

baseline_valid = valid["iv"].to_numpy()
ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
hgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", HistGradientBoostingRegressor(
        max_iter=120, learning_rate=0.05, max_leaf_nodes=15,
        l2_regularization=1.0, random_state=SEED
    ))
])

ridge.fit(X_train, y_train)
hgb.fit(X_train, y_train)

valid_results = [
    metric_row("Persistence", y_valid, valid["iv"]),
    metric_row("Ridge", y_valid, ridge.predict(X_valid)),
    metric_row("HistGradientBoosting", y_valid, hgb.predict(X_valid)),
]
valid_metrics = pd.DataFrame(valid_results).sort_values("RMSE")
display(valid_metrics)
best_name = valid_metrics.iloc[0]["model"]
print("Validation-selected model:", best_name)

## 9. Optional neural model (PyTorch)

A compact multilayer perceptron is the first neural baseline. It is deliberately small and uses early stopping based on validation loss. It does not claim to be the best architecture; only retain it if it beats suitable baselines on unseen data.

This cell runs only on the verified CUDA device and raises an error if the model or its tensors are not on it. There is no CPU fallback. It is skipped only when you set `RUN_NEURAL = False`.

In [ ]:
if torch is not None:
    import copy
    from torch import nn
    from torch.utils.data import TensorDataset, DataLoader

    DEVICE, HW_META = require_gpu()   # re-verify the selected GPU immediately before neural work
    device = DEVICE                   # verified CUDA device; there is no CPU fallback
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

    def assert_on_gpu(*objs):
        for o in objs:
            t = next(o.parameters()) if isinstance(o, nn.Module) else o
            if t.device != DEVICE:
                raise RuntimeError("Expected %s but found %s: CPU execution of neural computation is forbidden." % (DEVICE, t.device))

    _t0 = time.time()
    torch.cuda.reset_peak_memory_stats(device)

    # Scale using training data only.
    torch_imputer = SimpleImputer(strategy="median")
    Xtr_np = torch_imputer.fit_transform(X_train)
    Xva_np = torch_imputer.transform(X_valid)
    Xte_np = torch_imputer.transform(X_test)
    scaler = StandardScaler()
    Xtr_np = scaler.fit_transform(Xtr_np).astype(np.float32)
    Xva_np = scaler.transform(Xva_np).astype(np.float32)
    Xte_np = scaler.transform(Xte_np).astype(np.float32)

    y_mean = float(np.mean(y_train))
    y_std = float(np.std(y_train) + 1e-8)
    ytr_np = ((y_train.to_numpy(dtype=np.float32)-y_mean)/y_std).reshape(-1,1)
    yva_np = ((y_valid.to_numpy(dtype=np.float32)-y_mean)/y_std).reshape(-1,1)

    train_loader = DataLoader(
        TensorDataset(torch.tensor(Xtr_np), torch.tensor(ytr_np)),
        batch_size=512, shuffle=True
    )
    Xva_t = torch.tensor(Xva_np, device=device)
    yva_t = torch.tensor(yva_np, device=device)

    class IVMLP(nn.Module):
        def __init__(self, n_features):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(n_features, 64), nn.ReLU(), nn.Dropout(0.10),
                nn.Linear(64, 32), nn.ReLU(),
                nn.Linear(32, 1)
            )
        def forward(self, x):
            return self.net(x)

    nn_model = IVMLP(Xtr_np.shape[1]).to(device)
    assert_on_gpu(nn_model, Xva_t, yva_t)
    optimizer = torch.optim.AdamW(nn_model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.MSELoss()
    best_state, best_val, patience, stale = None, float("inf"), 12, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(100):
        nn_model.train()
        batch_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            assert_on_gpu(xb, yb)
            optimizer.zero_grad()
            loss = loss_fn(nn_model(xb), yb)
            loss.backward()
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu()))
        nn_model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(nn_model(Xva_t), yva_t).cpu())
        history["train_loss"].append(float(np.mean(batch_losses)))
        history["val_loss"].append(val_loss)
        if val_loss < best_val:
            best_val = val_loss
            best_state = copy.deepcopy(nn_model.state_dict())
            stale = 0
        else:
            stale += 1
        if stale >= patience:
            break

    nn_model.load_state_dict(best_state)
    nn_model.eval()
    with torch.no_grad():
        nn_valid_pred = nn_model(torch.tensor(Xva_np, device=device)).cpu().numpy().ravel()*y_std + y_mean
        nn_test_pred = nn_model(torch.tensor(Xte_np, device=device)).cpu().numpy().ravel()*y_std + y_mean

    torch.cuda.synchronize(device)
    NN_RUN = {   # recorded with every neural result (saved in the artifacts cell)
        "data_label": "SYNTHETIC - not market data" if DATA_MODE == "synthetic" else "user-supplied dataset",
        "model": "IVMLP 64-32-1 (ReLU, dropout 0.10)", "loss": "MSE on standardized target",
        "optimizer": "AdamW lr=1e-3 weight_decay=1e-4", "batch_size": 512, "max_epochs": 100, "early_stopping_patience": 12,
        "epochs_run": len(history["train_loss"]), "best_val_loss_standardized": best_val,
        "train_and_predict_seconds": round(time.time() - _t0, 2),
        "peak_gpu_memory_mib": round(torch.cuda.max_memory_allocated(device) / 2**20, 1),
        "device": str(device), "seed": SEED, "hardware": HW_META, "history": history,
    }
    # .cpu() above only exports GPU results to NumPy for the scikit-learn metrics; all tensor math ran on the GPU.

    plt.figure(figsize=(7,3))
    plt.plot(history["train_loss"], label="train")
    plt.plot(history["val_loss"], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel("Standardized MSE")
    plt.title("Neural baseline learning curve")
    plt.legend()
    plt.grid(True, alpha=.3)
    plt.show()

    display(pd.DataFrame([metric_row("PyTorch MLP", y_valid, nn_valid_pred)]))
else:
    print("RUN_NEURAL is False: neural cells skipped by explicit choice (no CPU fallback, no neural result).")

## 10. Final test evaluation

Choose the model using validation performance only. Then evaluate the selected baseline on the held-out test dates. The neural model is included only if trained and if you explicitly decide to compare it. Avoid repeatedly changing the model based on test metrics.

In [ ]:
# Choose the best non-neural model based on validation RMSE.
model_map = {"Ridge": ridge, "HistGradientBoosting": hgb}
selected = best_name if best_name in model_map else "Persistence"
if selected == "Persistence":
    test_pred = test["iv"].to_numpy()
else:
    test_pred = model_map[selected].predict(X_test)

test_predictions = pd.DataFrame({
    "date": test["date"].to_numpy(),
    "target_date": test["target_date"].to_numpy(),
    "underlying": test["underlying"].to_numpy(),
    "expiry": test["expiry"].to_numpy(),
    "strike": test["strike"].to_numpy(),
    "option_type": test["option_type"].to_numpy(),
    "actual_iv": y_test.to_numpy(),
    "persistence_pred": test["iv"].to_numpy(),
    "selected_model_pred": np.asarray(test_pred),
})
test_metrics = pd.DataFrame([
    metric_row("Persistence", y_test, test["iv"]),
    metric_row(selected, y_test, test_pred),
])
if torch is not None and "nn_test_pred" in globals():
    # Reported only: model selection above used validation data and excluded the MLP; nothing here is tuned on the test set.
    test_predictions["pytorch_mlp_pred"] = nn_test_pred
    test_metrics = pd.concat([test_metrics, pd.DataFrame([metric_row("PyTorch MLP (reported, not selected)", y_test, nn_test_pred)])], ignore_index=True)
display(test_metrics)
test_metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
test_predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)

## 11. Test error by surface region

Aggregate metrics can hide failures in wings or long-dated options. Inspect results by moneyness and maturity buckets. Buckets with too few observations should not be overinterpreted.

In [ ]:
region = test[["log_moneyness", "tau"]].copy()
region["actual_iv"] = y_test.to_numpy()
region["pred"] = np.asarray(test_pred)
region["abs_error"] = np.abs(region["pred"] - region["actual_iv"])
region["moneyness_bucket"] = pd.cut(region["log_moneyness"], [-np.inf, -0.15, -0.05, 0.05, 0.15, np.inf])
region["tau_bucket"] = pd.cut(region["tau"], [0, 30/365, 60/365, 120/365, np.inf],
                              labels=["<=30d", "30-60d", "60-120d", ">120d"])
display(region.groupby(["moneyness_bucket", "tau_bucket"], observed=False)["abs_error"]
        .agg(["count", "mean", "median"]).reset_index())

plt.figure(figsize=(7,4))
plt.scatter(test_predictions["actual_iv"], test_predictions["selected_model_pred"], s=8, alpha=.35)
lims = [
    min(test_predictions["actual_iv"].min(), test_predictions["selected_model_pred"].min()),
    max(test_predictions["actual_iv"].max(), test_predictions["selected_model_pred"].max())
]
plt.plot(lims, lims, "--")
plt.xlabel("Actual next-observation IV")
plt.ylabel("Predicted IV")
plt.title(f"Test predictions: {selected}")
plt.grid(True, alpha=.3)
plt.show()

## 12. Save model artifacts

The notebook saves the selected sklearn pipeline, imputer metadata, metrics, predictions, and (if trained) the neural model weights and preprocessing parameters. Keep a copy of the dataset version, provider license terms, and experiment configuration with any research result.

In [ ]:
# Save sklearn artifacts when applicable.
if selected in model_map:
    joblib.dump(model_map[selected], OUTPUT_DIR / f"{selected.lower()}_model.joblib")

if torch is not None and "nn_model" in globals():
    torch.save({
        "state_dict": nn_model.state_dict(),
        "feature_names": FEATURES,
        "y_mean": y_mean,
        "y_std": y_std,
        "seed": SEED,
        "device": str(DEVICE),
        "hardware": HW_META,
    }, OUTPUT_DIR / "pytorch_mlp_weights.pt")   # CUDA tensors: load with torch.load(..., map_location="cuda")
    joblib.dump({"imputer": torch_imputer, "scaler": scaler}, OUTPUT_DIR / "pytorch_preprocessing.joblib")
    with open(OUTPUT_DIR / "neural_run_metadata.json", "w") as f:
        json.dump(NN_RUN, f, indent=2, default=str)
    pd.DataFrame({
        "date": valid["date"].to_numpy(), "target_date": valid["target_date"].to_numpy(), "underlying": valid["underlying"].to_numpy(),
        "expiry": valid["expiry"].to_numpy(), "strike": valid["strike"].to_numpy(), "option_type": valid["option_type"].to_numpy(),
        "actual_iv": y_valid.to_numpy(), "persistence_pred": valid["iv"].to_numpy(), "pytorch_mlp_pred": nn_valid_pred,
    }).to_csv(OUTPUT_DIR / "neural_validation_predictions.csv", index=False)
    pd.DataFrame([metric_row("PyTorch MLP", y_valid, nn_valid_pred)]).to_csv(OUTPUT_DIR / "neural_validation_metrics.csv", index=False)

config = {
    "seed": SEED,
    "data_mode": DATA_MODE,
    "rows_after_cleaning": int(len(quotes)),
    "forecast_examples": int(len(dataset)),
    "train_rows": int(len(train)),
    "validation_rows": int(len(valid)),
    "test_rows": int(len(test)),
    "selected_validation_model": selected,
    "device": str(DEVICE) if DEVICE is not None else "none (RUN_NEURAL=False; scikit-learn baselines only)",
    "data_label": "SYNTHETIC - not market data" if DATA_MODE == "synthetic" else "user-supplied dataset",
    "hardware": HW_META,
    "preprocessing": {"imputer": "SimpleImputer(median), fit on train only", "scaler": "StandardScaler, fit on train only",
                      "target": "standardized with train mean/std (neural model)"},
    "neural_run": NN_RUN if "NN_RUN" in globals() else None,
    "features": FEATURES,
    "target": TARGET,
    "important_limitations": [
        "Synthetic demo data is not real-market validation.",
        "Target is next available observation for the same contract, not necessarily exactly one calendar day.",
        "Single chronological holdout is preliminary; perform walk-forward evaluation before research claims.",
        "No arbitrage-free surface constraints are imposed by this notebook.",
        "Real-data licensing and source assumptions must be verified by the user."
    ]
}
with open(OUTPUT_DIR / "run_config.json", "w") as f:
    json.dump(config, f, indent=2, default=str)

print("Saved artifacts to:", OUTPUT_DIR)
print(sorted(p.name for p in OUTPUT_DIR.iterdir()))

## 13. Optional: download outputs from Colab

Run this cell in Colab to create a ZIP of model artifacts and result files. If using Google Drive, you can instead copy `OUTPUT_DIR` to a folder you control.

In [ ]:
import shutil
zip_path = shutil.make_archive("/content/nvs_outputs", "zip", OUTPUT_DIR)
print("ZIP created:", zip_path)

# In Colab, uncomment to download:
# from google.colab import files
# files.download(zip_path)

## 14. Research checklist before trusting a result

- [ ] Data source and license permit the intended use.
- [ ] IV units are correct (decimal, not percent).
- [ ] Quote timestamp convention and option style are understood.
- [ ] Stale, crossed, zero-bid, and wide-spread quotes have been reviewed.
- [ ] Split is chronological and features use only information available at prediction time.
- [ ] Persistence and statistical baselines are reported.
- [ ] Hyperparameters are selected without using the final test period.
- [ ] Results are reported by maturity and moneyness, not only as one aggregate metric.
- [ ] Real-data results are not mixed with synthetic experiment results.
- [ ] The research does not claim profitability from forecast accuracy alone.

### Important next step for the project
Once the exact real dataset schema is known, adapt the column mapping and contract matching to the provider's conventions. This notebook is a complete runnable research scaffold, but real-data quality and market-specific conventions must be verified before treating it as a production-grade options research pipeline.